# Digits Image-Analysis Workflow

This notebook implements the full `fetch -> prepare -> analyze -> report` workflow for the `sklearn.datasets.load_digits` dataset. It is a complete worked example that writes the same style of stage artifacts used elsewhere in the repo, but it does the full class-summary and report work directly inside the notebook.


In [1]:
from __future__ import annotations

import json
import math
import os
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root from the current working directory.")


ROOT = find_repo_root()
os.chdir(ROOT)
NEWLINE = chr(10)

WORKFLOW_ROOT = ROOT / "runs" / "notebook-image-analysis"
FETCH_DIR = WORKFLOW_ROOT / "fetch"
PREPARE_DIR = WORKFLOW_ROOT / "prepare"
ANALYZE_DIR = WORKFLOW_ROOT / "analyze"
REPORT_DIR = WORKFLOW_ROOT / "report"

for directory in [FETCH_DIR, PREPARE_DIR, ANALYZE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_NPZ = FETCH_DIR / "digits_raw.npz"
IMAGES_NPY = PREPARE_DIR / "images.npy"
METADATA_CSV = PREPARE_DIR / "metadata.csv"
DATASET_OVERVIEW_JSON = ANALYZE_DIR / "dataset_overview.json"
CLASS_SUMMARY_CSV = ANALYZE_DIR / "class_image_summary.csv"
CLASS_REPRESENTATIVES_PNG = ANALYZE_DIR / "class_representatives.png"
REPORT_MD = REPORT_DIR / "report.md"

CLASS_SUMMARY_COLUMNS = [
    "label",
    "n_images",
    "mean_intensity",
    "std_intensity",
    "mean_edge_density",
]
EDGE_THRESHOLD = 0.2
NORMALIZE_DIVISOR = 16.0

print(f"Repo root: {ROOT}")
print(f"Notebook workflow output: {WORKFLOW_ROOT}")


Repo root: /home/can134/nextgen/nextgen2026-coding-bootcamp
Notebook workflow output: /home/can134/nextgen/nextgen2026-coding-bootcamp/runs/notebook-image-analysis


## Fetch

The fetch stage materializes the raw digits dataset into a stable `.npz` artifact so later steps can read from disk instead of from an in-memory loader.


In [2]:
digits = load_digits()
raw_images = digits.images.astype(np.float32)
raw_targets = digits.target.astype(np.int64)
raw_target_names = np.asarray(digits.target_names, dtype=np.int64)
raw_flat_data = digits.data.astype(np.float32)

np.savez_compressed(
    RAW_NPZ,
    images=raw_images,
    target=raw_targets,
    target_names=raw_target_names,
    data=raw_flat_data,
)

fetch_artifacts = {
    "dataset_name": "sklearn_digits",
    "raw_npz": str(RAW_NPZ),
    "n_images": int(raw_images.shape[0]),
    "n_classes": int(len(raw_target_names)),
    "image_shape": list(raw_images.shape[1:]),
}

fetch_artifacts


{'dataset_name': 'sklearn_digits',
 'raw_npz': '/home/can134/nextgen/nextgen2026-coding-bootcamp/runs/notebook-image-analysis/fetch/digits_raw.npz',
 'n_images': 1797,
 'n_classes': 10,
 'image_shape': [8, 8]}

## Prepare

The prepare stage normalizes the image values to the `[0, 1]` range and writes analysis-ready artifacts: an image array and a simple metadata table with `image_id` and `label`.


In [3]:
with np.load(RAW_NPZ) as raw_payload:
    images = raw_payload["images"].astype(np.float32) / NORMALIZE_DIVISOR
    labels = raw_payload["target"].astype(np.int64)

images = np.clip(images, 0.0, 1.0)
metadata = pd.DataFrame(
    {
        "image_id": np.arange(len(labels), dtype=int),
        "label": labels.astype(int),
    }
)

np.save(IMAGES_NPY, images)
metadata.to_csv(METADATA_CSV, index=False)

prepare_artifacts = {
    "images_npy": str(IMAGES_NPY),
    "metadata_csv": str(METADATA_CSV),
    "n_images": int(images.shape[0]),
    "image_shape": list(images.shape[1:]),
}

metadata.head()


,image_id,label
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


## Analyze

The analyze stage computes a dataset overview, a class-level summary artifact, and a representative-image montage. This is the full implementation of the workshop task rather than the scaffolded starter-state version.


In [4]:
def calculate_edge_density(image: np.ndarray, threshold: float) -> float:
    horizontal_edges = np.abs(np.diff(image, axis=1)) > threshold
    vertical_edges = np.abs(np.diff(image, axis=0)) > threshold
    total_edges = horizontal_edges.size + vertical_edges.size
    if total_edges == 0:
        return 0.0
    return float((horizontal_edges.sum() + vertical_edges.sum()) / total_edges)


def build_dataset_overview(images: np.ndarray, metadata: pd.DataFrame) -> dict:
    label_counts = metadata["label"].value_counts().sort_index()
    return {
        "dataset_name": "sklearn_digits",
        "n_images": int(len(metadata)),
        "n_classes": int(label_counts.shape[0]),
        "image_height": int(images.shape[1]),
        "image_width": int(images.shape[2]),
        "labels": [int(label) for label in label_counts.index.tolist()],
        "images_per_class": {str(int(label)): int(count) for label, count in label_counts.items()},
    }


def build_class_image_summary(
    images: np.ndarray,
    metadata: pd.DataFrame,
    edge_threshold: float,
) -> pd.DataFrame:
    rows = []
    for label in sorted(int(value) for value in metadata["label"].unique().tolist()):
        class_image_ids = metadata.loc[metadata["label"] == label, "image_id"].to_numpy(dtype=int)
        class_images = images[class_image_ids]
        image_mean_intensities = class_images.mean(axis=(1, 2))
        edge_densities = np.array(
            [calculate_edge_density(image=image, threshold=edge_threshold) for image in class_images],
            dtype=np.float64,
        )
        rows.append(
            {
                "label": label,
                "n_images": int(class_images.shape[0]),
                "mean_intensity": float(image_mean_intensities.mean()),
                "std_intensity": float(image_mean_intensities.std(ddof=0)),
                "mean_edge_density": float(edge_densities.mean()),
            }
        )
    return pd.DataFrame(rows, columns=CLASS_SUMMARY_COLUMNS)


def write_representative_image_montage(
    images: np.ndarray,
    metadata: pd.DataFrame,
    output_path: Path,
) -> None:
    labels = sorted(int(value) for value in metadata["label"].unique().tolist())
    ncols = 5
    nrows = math.ceil(len(labels) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4))
    axes = np.atleast_1d(axes).ravel()

    for ax, label in zip(axes, labels):
        class_image_ids = metadata.loc[metadata["label"] == label, "image_id"].to_numpy(dtype=int)
        class_images = images[class_image_ids]
        class_mean = class_images.mean(axis=0)
        distances = np.square(class_images - class_mean).sum(axis=(1, 2))
        representative = class_images[int(np.argmin(distances))]
        ax.imshow(representative, cmap="gray_r", vmin=0.0, vmax=1.0)
        ax.set_title(f"Digit {label}")
        ax.axis("off")

    for ax in axes[len(labels):]:
        ax.axis("off")

    fig.suptitle("Class representatives from notebook analyze stage", fontsize=12)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


dataset_overview = build_dataset_overview(images=images, metadata=metadata)
class_image_summary = build_class_image_summary(
    images=images,
    metadata=metadata,
    edge_threshold=EDGE_THRESHOLD,
)
write_representative_image_montage(
    images=images,
    metadata=metadata,
    output_path=CLASS_REPRESENTATIVES_PNG,
)

DATASET_OVERVIEW_JSON.write_text(json.dumps(dataset_overview, indent=2) + NEWLINE)
class_image_summary.to_csv(CLASS_SUMMARY_CSV, index=False)

analyze_artifacts = {
    "dataset_overview_json": str(DATASET_OVERVIEW_JSON),
    "class_image_summary_csv": str(CLASS_SUMMARY_CSV),
    "class_representatives_png": str(CLASS_REPRESENTATIVES_PNG),
    "edge_threshold": EDGE_THRESHOLD,
}

class_image_summary


,label,n_images,mean_intensity,std_intensity,mean_edge_density
0,0,178,0.309510,0.036673,0.463182
1,1,182,0.305884,0.044307,0.306662
2,2,177,0.306574,0.028881,0.397094
3,3,183,0.299645,0.032026,0.433694
4,4,181,0.303430,0.026065,0.437500
5,5,182,0.300025,0.028677,0.417533
6,6,181,0.303954,0.033157,0.412638
7,7,179,0.296182,0.028781,0.405676
8,8,174,0.322198,0.033337,0.429085
9,9,180,0.305946,0.034067,0.429911


## Report

The report stage reads the analyze artifacts, builds a `Digit Class Profiles` section from `class_image_summary.csv`, and writes a Markdown report that references the representative montage instead of recomputing anything.


In [5]:
def format_summary_for_markdown(summary_df: pd.DataFrame) -> pd.DataFrame:
    formatted = summary_df.copy()
    formatted["label"] = formatted["label"].astype(int).astype(str)
    formatted["n_images"] = formatted["n_images"].astype(int).astype(str)
    for column in ["mean_intensity", "std_intensity", "mean_edge_density"]:
        formatted[column] = formatted[column].map(lambda value: f"{value:.4f}")
    return formatted


def dataframe_to_markdown_table(df: pd.DataFrame) -> str:
    columns = list(df.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(str(row[column]) for column in columns) + " |" for _, row in df.iterrows()]
    return NEWLINE.join([header, separator, *rows])


def build_digit_class_profiles_section(class_summary_csv: Path) -> list[str]:
    summary_df = pd.read_csv(class_summary_csv)
    summary_table = dataframe_to_markdown_table(format_summary_for_markdown(summary_df))
    return [
        "## Digit Class Profiles",
        "",
        "The table below is rendered directly from `class_image_summary.csv` written by the analyze stage.",
        "",
        summary_table,
    ]


summary_df = pd.read_csv(CLASS_SUMMARY_CSV)
class_profile_section = build_digit_class_profiles_section(CLASS_SUMMARY_CSV)
representative_relpath = Path("../analyze") / CLASS_REPRESENTATIVES_PNG.name

report_lines = [
    "# Digits Workflow Report",
    "",
    "This report was generated from analyze-stage artifacts created inside the notebook workflow.",
    "",
    "## Dataset Overview",
    "",
    f"- Dataset: `{dataset_overview['dataset_name']}`",
    f"- Images analyzed: `{dataset_overview['n_images']}`",
    f"- Digit classes: `{dataset_overview['n_classes']}`",
    f"- Image shape: `{dataset_overview['image_height']}x{dataset_overview['image_width']}`",
    "",
    "## Analyze Artifacts",
    "",
    f"- `dataset_overview.json`: `{DATASET_OVERVIEW_JSON.name}`",
    f"- `class_image_summary.csv`: `{CLASS_SUMMARY_CSV.name}`",
    f"- `class_representatives.png`: `{CLASS_REPRESENTATIVES_PNG.name}`",
    "",
    "## Representative Digits",
    "",
    "The montage below is referenced from the analyze output rather than recomputed during reporting.",
    "",
    f"![Representative digits]({representative_relpath.as_posix()})",
    "",
    *class_profile_section,
]

REPORT_MD.write_text(NEWLINE.join(report_lines) + NEWLINE)
report_preview = REPORT_MD.read_text()
print(report_preview)


# Digits Workflow Report

This report was generated from analyze-stage artifacts created inside the notebook workflow.

## Dataset Overview

- Dataset: `sklearn_digits`
- Images analyzed: `1797`
- Digit classes: `10`
- Image shape: `8x8`

## Analyze Artifacts

- `dataset_overview.json`: `dataset_overview.json`
- `class_image_summary.csv`: `class_image_summary.csv`
- `class_representatives.png`: `class_representatives.png`

## Representative Digits

The montage below is referenced from the analyze output rather than recomputed during reporting.

![Representative digits](../analyze/class_representatives.png)

## Digit Class Profiles

The table below is rendered directly from `class_image_summary.csv` written by the analyze stage.

| label | n_images | mean_intensity | std_intensity | mean_edge_density |
| --- | --- | --- | --- | --- |
| 0 | 178 | 0.3095 | 0.0367 | 0.4632 |
| 1 | 182 | 0.3059 | 0.0443 | 0.3067 |
| 2 | 177 | 0.3066 | 0.0289 | 0.3971 |
| 3 | 183 | 0.2996 | 0.0320 | 0.4337 |

## Validation

The final cell checks the end-to-end contract and lists the artifacts written by the notebook workflow.


In [6]:
assert RAW_NPZ.exists()
assert IMAGES_NPY.exists()
assert METADATA_CSV.exists()
assert DATASET_OVERVIEW_JSON.exists()
assert CLASS_SUMMARY_CSV.exists()
assert CLASS_REPRESENTATIVES_PNG.exists()
assert REPORT_MD.exists()

validated_summary = pd.read_csv(CLASS_SUMMARY_CSV)
assert list(validated_summary.columns) == CLASS_SUMMARY_COLUMNS
assert validated_summary["label"].tolist() == list(range(10))
assert len(validated_summary) == 10

report_text = REPORT_MD.read_text()
assert "## Digit Class Profiles" in report_text
assert "class_image_summary.csv" in report_text

print("Notebook workflow artifacts:")
for path in sorted(WORKFLOW_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT))

validated_summary


Notebook workflow artifacts:
runs/notebook-image-analysis/analyze/class_image_summary.csv
runs/notebook-image-analysis/analyze/class_representatives.png
runs/notebook-image-analysis/analyze/dataset_overview.json
runs/notebook-image-analysis/fetch/digits_raw.npz
runs/notebook-image-analysis/prepare/images.npy
runs/notebook-image-analysis/prepare/metadata.csv
runs/notebook-image-analysis/report/report.md


,label,n_images,mean_intensity,std_intensity,mean_edge_density
0,0,178,0.309510,0.036673,0.463182
1,1,182,0.305884,0.044307,0.306662
2,2,177,0.306574,0.028881,0.397094
3,3,183,0.299645,0.032026,0.433694
4,4,181,0.303430,0.026065,0.437500
5,5,182,0.300025,0.028677,0.417533
6,6,181,0.303954,0.033157,0.412638
7,7,179,0.296182,0.028781,0.405676
8,8,174,0.322198,0.033337,0.429085
9,9,180,0.305946,0.034067,0.429911
